In [ ]:
## medication status of participants

In [ ]:
#%%bash
#conda install openpyxl

In [ ]:
import pandas as pd
import re
from collections import Counter
import openpyxl

In [ ]:
df = pd.read_excel("medication_analysis_by_dataset.xlsx")
df.head()

,Bids-Number,Antidepressants,Group,Gender,Age,Medication,Dataset,AD_Classes,Has_AD
0,Sub-007,1.0,Patient,male,23,morning: Citalopram 30mg,EEG,['SSRI'],True
1,Sub-008,1.0,Patient,female,19,morning: Escitalopram 10mg evening: Quetiapin...,EEG,['SSRI'],True
2,Sub-009,1.0,Patient,male,27,morning: Elontril 300mg,EEG,['NDRI'],True
3,Sub-012,1.0,Patient,male,36,"morning: Amlopipin 5mg, Bisoprlop 5mg, Elontri...",EEG,['NDRI'],True
4,Sub-020,1.0,Patient,female,32,morning: Cipralex 10mg,EEG,['SSRI'],True


In [ ]:
# --- Extract AD classes from medication text ---
def extract_ad_classes(medication_str):
    if pd.isna(medication_str):
        return []
    med_str = medication_str.lower()
    classes = []
    if re.search(r'citalopram|escitalopram|cipralex', med_str):
        classes.append('SSRI')
    if re.search(r'bupropion|elontril', med_str):
        classes.append('NDRI')
    if re.search(r'venla|duloxetine|milnacipran', med_str):
        classes.append('SNRI')
    if re.search(r'amitriptylin|trimipramin|clomipramin', med_str):
        classes.append('TCA')
    if re.search(r'mirtazapin', med_str):
        classes.append('NaSSA')
    if re.search(r'monoaminoxidase|tranylcypromine|moclobemid', med_str):
        classes.append('MAOI')
    return list(set(classes))  # ensure uniqueness

# --- Extract AD substances ---
def extract_ad_substances(med_str):
    if pd.isna(med_str):
        return []
    med_str = med_str.lower()
    substances = []
    for name in ['citalopram', 'escitalopram', 'cipralex',
                 'elontril', 'bupropion',
                 'venlafaxine', 'duloxetine', 'milnacipran',
                 'amitriptylin', 'trimipramin', 'clomipramin',
                 'mirtazapin',
                 'tranylcypromine', 'moclobemid']:
        if name in med_str:
            substances.append(name)
    return list(set(substances))

# --- Apply classification ---
df['AD_Classes'] = df['Medication'].apply(extract_ad_classes)
df['AD_Substances'] = df['Medication'].apply(extract_ad_substances)
df['Has_AD'] = df['AD_Classes'].apply(lambda x: len(x) > 0)
df['Num_AD_Classes'] = df['AD_Classes'].apply(len)
df['Num_AD_Substances'] = df['AD_Substances'].apply(len)

# --- Compute base stats ---
total = len(df)
with_ad = df['Has_AD'].sum()
without_ad = total - with_ad
percent_with = with_ad / total * 100
percent_without = without_ad / total * 100
all_classes = sum(df['AD_Classes'], [])
class_counts = Counter(all_classes)
class_percentages = {k: v / with_ad * 100 for k, v in class_counts.items()}

# --- Additional insights ---
multi_class_count = (df['Num_AD_Classes'] > 1).sum()
multi_substance_count = (df['Num_AD_Substances'] > 1).sum()

# --- Print Output ---
print("\n===== FULL SAMPLE ANALYSIS =====")
print(f"Total Subjects:                            {total}")
print(f"With Antidepressants:                      {percent_with:.1f}%")
print(f"Without Antidepressants:                   {percent_without:.1f}%")

print("\nAntidepressant Class Breakdown (among AD users):")
for cls, percent in class_percentages.items():
    print(f"  - {cls}: {percent:.1f}%")

print(f"\nSubjects using >1 AD class:                {multi_class_count} ({multi_class_count / with_ad * 100:.1f}%)")
print(f"Subjects using >1 AD substance (any class): {multi_substance_count} ({multi_substance_count / with_ad * 100:.1f}%)")





===== FULL SAMPLE ANALYSIS =====
Total Subjects:                            107
With Antidepressants:                      44.9%
Without Antidepressants:                   55.1%

Antidepressant Class Breakdown (among AD users):
  - SSRI: 54.2%
  - NDRI: 12.5%
  - SNRI: 29.2%
  - NaSSA: 22.9%

Subjects using >1 AD class:                9 (18.8%)
Subjects using >1 AD substance (any class): 9 (18.8%)


In [24]:
# Show single-class usage (independent counts)
single_class_percentages = {
    cls: round(sum(cls in classes for classes in df[df['Has_AD']]['AD_Classes']) / with_ad * 100, 1)
    for cls in set(sum(df['AD_Classes'], []))
}

# Show exact class combinations (mutually exclusive)
from collections import Counter
def combo_label(classes): return " + ".join(sorted(classes))
df['Class_Combo'] = df['AD_Classes'].apply(combo_label)
combo_counts = df[df['Has_AD']]['Class_Combo'].value_counts()
combo_percentages = (combo_counts / with_ad * 100).round(1)

# Print structured result
print("\nAntidepressant Class Usage (among AD users):")
for cls, pct in sorted(single_class_percentages.items()):
    print(f"- {cls}: {pct:.1f}%")

print("\nCombination of Antidepressant Classes (exclusive combinations):")
for combo, pct in combo_percentages.items():
    print(f"-- {combo}: {pct:.1f}%")



Antidepressant Class Usage (among AD users):
- NDRI: 12.5%
- NaSSA: 22.9%
- SNRI: 29.2%
- SSRI: 54.2%

Combination of Antidepressant Classes (exclusive combinations):
-- SSRI: 39.6%
-- SNRI: 25.0%
-- NaSSA + SSRI: 14.6%
-- NDRI: 12.5%
-- NaSSA: 4.2%
-- NaSSA + SNRI: 4.2%
